# Gemini API: Summarize large documents using LangChain

## Overview

The [Gemini](https://ai.google.dev/models/gemini) models are a family of generative AI models that allow developers generate content and solve problems. These models are designed and trained to handle both text and images as input.

[LangChain](https://www.langchain.com/) is a framework designed to make integration of Large Language Models (LLM) like Gemini easier for applications.

In this notebook, you'll learn how to create an application to summarize large documents using the Gemini API and LangChain.


## Setup

### Install the Google GenAI SDK

Install the Google GenAI SDK from [npm](https://www.npmjs.com/package/@google/genai). 

```bash
$ npm install @google/genai
```

### Setup your API key

You can [create](https://aistudio.google.com/app/apikey) your API key using Google AI Studio with a single click.

Remember to treat your API key like a password. Don't accidentally save it in a notebook or source file you later commit to GitHub. In this notebook we will be storing the API key in a `.env` file. You can also set it as an environment variable or use a secret manager. 

Here's how to set it up in a `.env` file:

```bash
$ touch .env
$ echo "GEMINI_API_KEY=<YOUR_API_KEY>" >> .env
```

:::{.callout-tip}

Another option is to set the API key as an environment variable. You can do this in your terminal with the following command:

```bash
$ export GEMINI_API_KEY="<YOUR_API_KEY>"
```
:::

### Load the API key

To load the API key from the `.env` file, we will use the `dotenv` package. This package loads environment variables from a `.env` file into `process.env`. 

```bash
$ npm install dotenv
```

Then, we can load the API key in our code:


In [1]:
const dotenv = require("dotenv") as typeof import("dotenv");

dotenv.config({
  path: "../../.env",
});

const GEMINI_API_KEY = process.env.GEMINI_API_KEY ?? "";
if (!GEMINI_API_KEY) {
  throw new Error("GEMINI_API_KEY is not set in the environment variables");
}
console.log("GEMINI_API_KEY is set in the environment variables");


GEMINI_API_KEY is set in the environment variables


:::{.callout-note}
In our particular case the `.env` is is two directories up from the notebook, hence we need to use `../../` to go up two directories. If the `.env` file is in the same directory as the notebook, you can omit it altogether.

```
│
├── .env
└── examples
    └── langchain
        └── Gemini_LangChain_Summarization_WebLoad.ipynb
```
:::


### Select a model

Now select the model you want to use in this guide, either by selecting one in the list or writing it down. Keep in mind that some models, like the 2.5 ones are thinking models and thus take slightly more time to respond (cf. [thinking notebook](../../quickstarts/Get_started_thinking.ipynb) for more details and in particular learn how to switch the thiking off).


In [2]:
const tslab = require("tslab") as typeof import("tslab");

const MODEL_ID = "gemini-2.5-flash-preview-05-20";


## Summarize text

In this tutorial, you are going to summarize the text from a website using the Gemini model integrated through LangChain.

You'll perform the following steps to achieve the same:

1. Read and parse the website data using LangChain.
2. Chain together the following:
    - A prompt for extracting the required input data from the parsed website data.
    - A prompt for summarizing the text using LangChain.
    - An LLM model (such as the Gemini model) for prompting.
3. Run the created chain to prompt the model for the summary of the website data.


### Read and parse the website data

LangChain provides a wide variety of document loaders. To read the website data as a document, you will use the `WebBaseLoader` from LangChain.

To know more about how to read and parse input data from different sources using the document loaders of LangChain, read LangChain's [document loaders guide](https://python.langchain.com/docs/integrations/document_loaders).


In [3]:
import { CheerioWebBaseLoader } from "@langchain/community/document_loaders/web/cheerio";

const loader = new CheerioWebBaseLoader("https://blog.google/technology/ai/google-gemini-ai/");
const docs = await loader.load();


### Initialize the Gemini model

You must import the `ChatGoogleGenerativeAI` LLM from LangChain to initialize your model.

In this example you will use Gemini 2.0 Flash, (`gemini-2.5-flash`), as it supports text summarization. To know more about this model and the other models availabe, read Google AI's [language documentation](https://ai.google.dev/models/gemini).

You can configure the model parameters such as `temperature` or `topP`, by passing the appropriate values when creating the `ChatGoogleGenerativeAI` LLM. To learn more about the parameters and their uses, read Google AI's [concepts guide](https://ai.google.dev/docs/concepts#model_parameters).


In [4]:
import { ChatGoogleGenerativeAI } from "@langchain/google-genai";

const llm = new ChatGoogleGenerativeAI({
  model: MODEL_ID,
  apiKey: GEMINI_API_KEY,
});


### Create prompt templates

You'll use LangChain's `PromptTemplate` to generate prompts for summarizing the text.

To summarize the text from the website, you will need the following prompts.

1. Prompt to extract the data from the output of `WebBaseLoader`, named `doc_prompt`
2. Prompt for the Gemini model to summarize the extracted text, named `llm_prompt`.

In the `llm_prompt`, the variable `text` will be replaced later by the text from the website.


In [7]:
import { PromptTemplate } from "@langchain/core/prompts";

const doc_prompt = PromptTemplate.fromTemplate("{page_content}");
const llm_prompt = PromptTemplate.fromTemplate(`
    Write a concise summary of the following:
    "{text}"
    CONCISE SUMMARY:`);


### Create a stuff documents chain

LangChain provides [Chains](https://js.langchain.com/docs/how_to/sequence/) for chaining together LLMs with each other or other components for complex applications. You will create a **stuff documents chain** for this application. A stuff documents chain lets you combine all the relevant documents, insert them into the prompt, and pass that prompt to the LLM.

You can create a stuff documents chain using the [LangChain Expression Language (LCEL)](https://js.langchain.com/docs/expression_language).

To learn more about different types of document chains, read LangChain's [chains guide](https://v03.api.js.langchain.com/modules/langchain.chains.html).


In [28]:
import { Document } from "@langchain/core/documents";
import { RunnableSequence, RunnableLambda } from "@langchain/core/runnables";
import { StringOutputParser } from "@langchain/core/output_parsers";

const stuff_chain = RunnableSequence.from([
  new RunnableLambda({
    func: async (docs: Document[]) => {
      const text = await Promise.all(
        docs.map(async (doc) => await doc_prompt.format({ page_content: doc.pageContent }))
      );
      return { text: text.join("\n\n") };
    },
  }),
  llm_prompt,
  llm,
  new StringOutputParser(),
]);


### Prompt the model

To generate the summary of the the website data, pass the documents extracted using the `WebBaseLoader` (`docs`) to `invoke()`.


In [29]:
tslab.display.markdown(await stuff_chain.invoke(docs));


Google has launched Gemini, its largest and most capable AI model, designed to be natively multimodal (understanding text, code, audio, image, and video). Available in three sizes—Ultra (most complex tasks), Pro (broad use), and Nano (on-device)—Gemini achieves state-of-the-art performance across various benchmarks, including sophisticated reasoning and advanced coding. Developed with a strong emphasis on responsibility and safety, Gemini Pro is now integrated into Bard and will expand to other Google products like Search, while Gemini Nano powers new features on Pixel 8 Pro. Developers can access Gemini Pro via API starting December 13th, with Gemini Ultra to be released more broadly early next year after extensive safety checks.

## Conclusion

That's it. You have successfully created an LLM application to summarize text using LangChain and the Gemini API.
